# 04 — 보정계수 (α) 실험

`03_experiments.ipynb`의 `[1]` `[2]`를 그대로 가져왔다. 캐시(`{OUT}/{name}_samples.npz`)가
있으면 즉시 로드되므로 03을 먼저 돌릴 필요는 없다.

**작업 순서 (인수인계 2일차 6절)**
1. 3지역(대전·홍천·순천)에서 α 보정이 되는지 확인  ← 이 노트북
2. 지역을 6개로 늘려 α가 지역마다 흩어지는지 확인

**주의 — 탄소계수**
03의 `[R]` `[5]` `[6]` `[16]`은 전부 구계수 `{1:55, 2:65, 3:60}`으로 되돌린다.
이 노트북은 `[A]`에서 확정계수(79.25 / 101.02 / 92.01)를 명시한다.
따라서 03의 탄소 수치와 직접 비교하면 안 되고, **F1로 대조**해야 한다.

> ⚠️ **출력 해석 주의 (2026-10-02 추가, 인수인계 5일차 7-1·8-1절)**
>
> 이 노트북은 **1지역 학습 → 1지역 평가(쌍별) 설계**로 돌린 것이다. 이 설계는 폐기됐다.
>
> - **폐기 — 근거로 쓰지 말 것:** α\* 6개 값, "학습 지역 구조 / 군간·군내 비", α\*–위도 관계, "6지역 확장에서 이 구조가 유지되는지" 문구
> - **쌍별 설계 기준으로만 표시:** LOO 0.494 ± 0.158 % (LORO에서 재측정 예정)
> - **판정 기준으로 쓰지 말 것:** NOISE 0.239 %p (단위 근거 불명)
> - **유효:** 매개화 `shift()`와 검산 6/6, F1 개선 +0.036 (30/30 양수), 하한 3.094 ± 0.157 %
>
> `shift()`, `carbon_err`, `alpha_star`는 이후 실험에서 재사용한다.

In [ ]:
# ============================================================
# [1] 셋업 및 지역별 샘플 추출
#   전체 이미지를 메모리에 올리지 않고 블록 단위로 읽으며
#   학습/평가에 쓸 픽셀만 뽑아 npz로 저장한다.
#   → 지역 크기와 무관하게 동작하고, 이후 실험이 즉시 로드된다
# ============================================================
!pip install -q rasterio

import numpy as np, rasterio, os
from rasterio.windows import Window
from scipy.ndimage import uniform_filter
from google.colab import drive

drive.mount('/content/drive')
D   = '/content/drive/MyDrive/forest/data'
OUT = '/content/drive/MyDrive/forest/outputs'
os.makedirs(OUT, exist_ok=True)
names_cls = ['침엽수림', '활엽수림', '혼효림']

def make_feats(X):
    """(20,h,w) 반사율 → (38,h,w) 피처. 01_explore 와 동일한 구성/순서"""
    nd = lambda a, b: (a - b) / (a + b + 1e-6)
    s_ndvi, w_ndvi = nd(X[6], X[2]),  nd(X[16], X[12])
    s_ndmi, w_ndmi = nd(X[6], X[8]),  nd(X[16], X[18])
    d_ndvi, d_ndmi = s_ndvi - w_ndvi, s_ndmi - w_ndmi
    f = np.concatenate([X, np.stack([s_ndvi, w_ndvi, s_ndmi,
                                     w_ndmi, d_ndvi, d_ndmi])])
    extra = []
    for base in (s_ndvi, w_ndvi, d_ndvi):
        for w in (5, 15):
            a  = np.nan_to_num(base)
            mu = uniform_filter(a, size=w)
            sd = np.sqrt(np.maximum(uniform_filter(a**2, size=w) - mu**2, 0))
            extra += [mu, sd]
    return np.concatenate([f, np.stack(extra)]).astype('float32')


def extract_samples(name, n_max=600_000, block=1024, pad=16, seed=0, TILE=500):
    """지역에서 유효 픽셀(산림 3클래스, 갱신년도 2024+)을 최대 n_max개 샘플링"""
    cache = f'{OUT}/{name}_samples.npz'
    if os.path.exists(cache):
        z = np.load(cache)
        print(f'{name}: 캐시 로드 {z["X"].shape}')
        return {k: z[k] for k in z.files}

    with rasterio.open(f'{D}/{name}_label.tif')   as s: label = s.read(1)
    with rasterio.open(f'{D}/{name}_year.tif')    as s: year  = s.read(1)
    with rasterio.open(f'{D}/{name}_species.tif') as s: sp    = s.read(1)

    H, W = label.shape
    use  = (label >= 1) & (label <= 3) & (year >= 2024)
    frac = min(1.0, n_max / max(use.sum(), 1))
    rng_b = np.random.default_rng(seed)
    Xs, ys, ts, ss = [], [], [], []

    with rasterio.open(f'{D}/s2/{name}_s2_20band.tif') as src:
        scale = 10000.0 if src.dtypes[0] == 'int16' else 1.0
        for r0 in range(0, H, block):
            for c0 in range(0, W, block):
                r1, c1 = min(H, r0+block), min(W, c0+block)
                m = use[r0:r1, c0:c1]
                if not m.any(): continue
                sel = m & (rng_b.random(m.shape) < frac)
                if not sel.any(): continue

                pr0, pc0 = max(0, r0-pad), max(0, c0-pad)
                pr1, pc1 = min(H, r1+pad), min(W, c1+pad)
                X = src.read(window=Window(pc0, pr0, pc1-pc0, pr1-pr0))
                X = X.astype('float32') / scale
                f = make_feats(X)
                f = f[:, r0-pr0:r0-pr0+(r1-r0), c0-pc0:c0-pc0+(c1-c0)]

                rr, cc = np.where(sel)
                Xs.append(f[:, sel].T)
                ys.append(label[r0:r1, c0:c1][sel])
                ss.append(sp[r0:r1, c0:c1][sel])
                ts.append(((rr+r0)//TILE)*10000 + (cc+c0)//TILE)
                del X, f

    d = dict(X=np.concatenate(Xs), y=np.concatenate(ys),
             tile=np.concatenate(ts), sp=np.concatenate(ss))
    np.savez_compressed(cache, **d)
    print(f'{name}: {d["X"].shape[0]:,}개 추출 → 저장')
    return d

Mounted at /content/drive


In [ ]:
# ============================================================
# [2] 지역별 샘플 추출 (첫 실행만 오래 걸림, 이후 캐시)
#   홍천 1.6GB 를 블록 단위로 읽으므로 5~10분 소요 가능
# ============================================================
S = {n: extract_samples(n) for n in ['daejeon', 'hongcheon', 'suncheon']}
for n, d in S.items():
    u, c = np.unique(d['y'], return_counts=True)
    print(f'{n:10s} {d["X"].shape[0]:>8,}개 | '
          f'침엽 {c[0]:>7,} 활엽 {c[1]:>7,} 혼효 {c[2]:>7,}')

daejeon: 캐시 로드 (599719, 38)
hongcheon: 캐시 로드 (599527, 38)
suncheon: 캐시 로드 (600846, 38)
daejeon     599,719개 | 침엽 239,240 활엽 238,972 혼효 121,507
hongcheon   599,527개 | 침엽 225,716 활엽 294,060 혼효  79,751
suncheon    600,846개 | 침엽 311,835 활엽 227,890 혼효  61,121


## `[A]` α 보정 스윕

`[16]`은 학습·평가 양쪽을 각자 통계로 z-score 했다.
이 셀은 **학습 지역을 그대로 두고 평가 지역만 α만큼 끌어온다.**

    med_α = α·med_a + (1−α)·med_b
    X'_b  = med_α + (X_b − med_b) · (sd_a/sd_b)^α

- α=0 → 원본
- α=1 → `[16]`의 norm과 **수학적으로 동일** (RF는 피처별 단조 아핀 변환에 불변)
- 학습 지역을 안 건드리므로 **RF 학습 3회**, α마다 predict만

**내장 검산**: α=0 / α=1의 macro F1이 `[16]` 출력을 재현해야 한다(허용 0.005).
여기서 틀리면 아래 해석은 전부 무효다.

소요 8~12분.

In [ ]:
# ============================================================
# [22] α 보정 스윕 — 3지역
#
#   [16]은 학습·평가 양쪽을 각자 통계로 z-score 했고, 전이 6조합 전부
#   부호가 반전했다(평균 1.9배 과보정). 보정을 끄고 켜는 게 아니라
#   보정량을 α 로 두고 0~1 사이를 훑는다.
#
#     med_α = α·med_a + (1−α)·med_b
#     X'_b  = med_α + (X_b − med_b) · (sd_a/sd_b)^α
#
#   α=0 → 원본.  α=1 → [16]의 norm 과 수학적으로 동일(RF는 피처별
#   단조 아핀 변환에 불변). 따라서 α=0, α=1 의 F1 이 [16] 출력을
#   재현해야 한다 — 내장 검산.
#
#   학습 지역은 건드리지 않으므로 RF 학습은 지역당 1회, 총 3회.
#   나머지는 predict 뿐이라 α 를 여러 점 봐도 비용이 거의 안 는다.
#
#   선행: [1] → [2].  소요 8~12분.
# ============================================================
import numpy as np, itertools
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

REGIONS = list(S)                                 # 03은 이 변수를 더 뒤에서 정의한다
SEED, N_TR, N_TE = 16, 200_000, 150_000          # [16]과 동일
ALPHAS = [0.0, 0.25, 0.40, 0.50, 0.60, 0.75, 1.0]

C     = {1: 79.25, 2: 101.02, 3: 92.01}          # 확정계수 ([20])
NOISE = 0.11 * (C[2] - C[1]) / 10.0              # 0.239 %p
print(f'C = {C}   NOISE = {NOISE:.3f}%p')

def carbon_err(y_true, y_pred):
    t = sum((y_true == v).sum() * C[v] for v in (1, 2, 3))
    p = sum((y_pred == v).sum() * C[v] for v in (1, 2, 3))
    return (p - t) * 100 / t

# ---------- 지역 통계 (라벨 불필요) ----------
STAT = {r: (np.median(S[r]['X'], axis=0), S[r]['X'].std(axis=0) + 1e-8)
        for r in REGIONS}

def shift(b, idx, a, alpha):
    """평가 지역 b 의 피처를 학습 지역 a 쪽으로 alpha 만큼 이동"""
    if a == b or alpha == 0.0:
        return S[b]['X'][idx]
    mb, sb = STAT[b]
    ma, sa = STAT[a]
    return ((alpha * ma + (1 - alpha) * mb)
            + (S[b]['X'][idx] - mb) * (sa / sb) ** alpha).astype('float32')

# ---------- 분할: [16]과 동일한 난수 순서 ----------
IDX = {}
for rg in REGIONS:
    r = np.random.default_rng(SEED)
    tiles = np.unique(S[rg]['tile'])
    te_t  = set(r.choice(tiles, int(len(tiles) * 0.3), replace=False).tolist())
    is_te = np.isin(S[rg]['tile'], list(te_t))
    IDX[rg] = {
        'tr':      r.choice(np.where(~is_te)[0], min(N_TR, (~is_te).sum()), replace=False),
        'te_self': r.choice(np.where(is_te)[0],  min(N_TE, is_te.sum()),   replace=False),
        'te_all':  r.choice(len(S[rg]['y']), N_TE, replace=False),
    }

# ---------- 실행 ----------
res = {}                                          # res[(a,b)][alpha] = (f1, carbon)
for a in REGIONS:
    clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                 max_features='sqrt', class_weight='balanced',
                                 n_jobs=-1, random_state=SEED)
    clf.fit(S[a]['X'][IDX[a]['tr']], S[a]['y'][IDX[a]['tr']])
    for b in REGIONS:
        te = IDX[b]['te_self'] if a == b else IDX[b]['te_all']
        y  = S[b]['y'][te]
        res[(a, b)] = {}
        for al in ALPHAS:
            p = clf.predict(shift(b, te, a, al))
            res[(a, b)][al] = (f1_score(y, p, average='macro', labels=[1, 2, 3]),
                               carbon_err(y, p))
            if a == b and al > 0:                 # 지역 내는 α 무관 → 첫 값 복사
                res[(a, b)][al] = res[(a, b)][0.0]
                break
    print(f'  {a[:3]} 완료')

# ---------- 표: 탄소 오차 ----------
CROSS = [(a, b) for a in REGIONS for b in REGIONS if a != b]
print('\n탄소 오차 (%)')
print(f'{"조합":12s}' + ''.join(f'{f"α={al:g}":>9s}' for al in ALPHAS))
for a, b in CROSS:
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{res[(a,b)][al][1]:+9.2f}' for al in ALPHAS))
print(f'{"|평균|":12s}' + ''.join(
    f'{np.mean([abs(res[k][al][1]) for k in CROSS]):9.2f}' for al in ALPHAS))

print('\nmacro F1')
print(f'{"조합":12s}' + ''.join(f'{f"α={al:g}":>9s}' for al in ALPHAS))
for a, b in CROSS:
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{res[(a,b)][al][0]:9.3f}' for al in ALPHAS))

# ---------- 검산: α=0 / α=1 이 [16] 재현 ----------
REF = {('daejeon','hongcheon'): (0.566, 0.533), ('daejeon','suncheon'): (0.540, 0.589),
       ('hongcheon','daejeon'): (0.582, 0.565), ('hongcheon','suncheon'): (0.546, 0.530),
       ('suncheon','daejeon'):  (0.588, 0.591), ('suncheon','hongcheon'): (0.485, 0.490)}
print('\n[검산] F1 이 [16]의 raw / norm 열과 일치해야 함 (허용 0.005)')
bad = 0
for k, (r0, r1) in REF.items():
    d0, d1 = res[k][0.0][0] - r0, res[k][1.0][0] - r1
    ok = abs(d0) < 0.005 and abs(d1) < 0.005
    bad += not ok
    print(f'  {k[0][:3]}→{k[1][:3]}  α=0 {res[k][0.0][0]:.3f} vs {r0:.3f} (Δ{d0:+.3f})   '
          f'α=1 {res[k][1.0][0]:.3f} vs {r1:.3f} (Δ{d1:+.3f})  {"O" if ok else "X"}')
print(f'  → {6-bad}/6 일치' + ('' if bad == 0 else '   ★ 불일치 시 아래 해석 무효'))

# ---------- α* : 부호가 뒤집히는 지점 ----------
print('\n[α*] 탄소 오차가 0을 지나는 α (선형보간)')
astar = {}
for a, b in CROSS:
    xs = ALPHAS; ys = [res[(a,b)][al][1] for al in xs]
    hit = next((i for i in range(len(xs)-1) if ys[i]*ys[i+1] < 0), None)
    if hit is None:
        astar[(a,b)] = np.nan
        print(f'  {a[:3]}→{b[:3]:8s} 구간 내 부호 반전 없음 '
              f'({ys[0]:+.2f} → {ys[-1]:+.2f})')
    else:
        t = xs[hit] + (xs[hit+1]-xs[hit]) * abs(ys[hit])/(abs(ys[hit])+abs(ys[hit+1]))
        astar[(a,b)] = t
        print(f'  {a[:3]}→{b[:3]:8s} α* = {t:.3f}')

v = np.array([astar[k] for k in CROSS], float)
if np.isfinite(v).sum() >= 4:
    def spread(by):
        g = {}
        for k, x in zip(CROSS, v):
            g.setdefault(by(k), []).append(x)
        within = np.nanmean([np.nanstd(x) for x in g.values()])
        between = np.nanstd([np.nanmean(x) for x in g.values()])
        return within, between
    wa, ba = spread(lambda k: k[0])
    wb, bb = spread(lambda k: k[1])
    print(f'\n[구조] α* 전체 평균 {np.nanmean(v):.3f}  표준편차 {np.nanstd(v):.3f}')
    print(f'  학습 지역으로 묶을 때  군내 {wa:.3f} / 군간 {ba:.3f}  → 비 {ba/(wa+1e-9):.1f}')
    print(f'  평가 지역으로 묶을 때  군내 {wb:.3f} / 군간 {bb:.3f}  → 비 {bb/(wb+1e-9):.1f}')
    print('  학습 지역 비가 3 이상이면 α 는 학습 지역만으로 정해진다')
    print('  → 타깃 라벨 없이 추정 가능. 6지역 확장에서 이 구조가 유지되는지가 다음 검사')
    print('  둘 다 3 미만이면 α 는 조합마다 제각각 → 타깃 소량 라벨이 필요하다는 결과')

# ---------- 3종 비교 ----------
print('\n[3종 비교] 평균 |탄소오차| (%)')
lo = np.mean([abs(res[k][0.0][1]) for k in CROSS])
best_single = min(ALPHAS, key=lambda al: np.mean([abs(res[k][al][1]) for k in CROSS]))
mid = np.mean([abs(res[k][best_single][1]) for k in CROSS])
up = np.mean([min(abs(res[k][al][1]) for al in ALPHAS) for k in CROSS])
print(f'  하한  보정 없음 (α=0)              {lo:6.2f}')
print(f'  본방식 전 조합 공통 α={best_single:g}         {mid:6.2f}')
print(f'  상한  조합별 최적 α (타깃 라벨 필요) {up:6.2f}')
print(f'  → 본방식이 하한보다 나쁘면 공통 α 는 성립하지 않는다')

C = {1: 79.25, 2: 101.02, 3: 92.01}   NOISE = 0.239%p
  dae 완료
  hon 완료
  sun 완료

탄소 오차 (%)
조합                α=0   α=0.25    α=0.4    α=0.5    α=0.6   α=0.75      α=1
dae→hon         +2.24    +1.37    +0.77    +0.35    -0.09    -0.83    -2.31
dae→sun         -2.27    -1.20    -0.56    -0.13    +0.29    +0.91    +1.90
hon→dae         -1.30    -0.20    +0.47    +0.91    +1.35    +1.96    +2.91
hon→sun         -3.17    -0.78    +0.59    +1.44    +2.20    +3.24    +4.62
sun→dae         +3.06    +2.13    +1.58    +1.20    +0.84    +0.28    -0.73
sun→hon         +5.93    +4.59    +3.63    +2.88    +2.03    +0.50    -2.66
|평균|             3.00     1.71     1.27     1.15     1.13     1.29     2.52

macro F1
조합                α=0   α=0.25    α=0.4    α=0.5    α=0.6   α=0.75      α=1
dae→hon         0.566    0.574    0.578    0.576    0.573    0.564    0.533
dae→sun         0.540    0.576    0.588    0.595    0.598    0.598    0.589
hon→dae         0.582    0.595    0.598    0.597    0.593    0

In [ ]:
# ============================================================
# [B] 시드 반복 — α* 와 F1 개선이 난수에 견디는가
#
#   [A]는 SEED=16 한 번이다. 1일차에 F1 0.556→0.542 를 1회 실행이라
#   폐기한 기준을 그대로 적용하면 [A]의 F1 개선(0.551→0.586)도 아직 못 쓴다.
#
#   시드가 바꾸는 것: 타일 분할, 학습 표본 추출, RF 자체.
#   세 가지를 함께 흔든다 = 실제로 마주할 변동.
#
#   보는 것
#     1. α* 가 시드에 안정한가 (학습 지역별 sd)
#     2. 학습 지역 구조(군간/군내 비 10.0)가 유지되는가
#     3. F1 개선이 모든 시드 × 모든 조합에서 같은 부호인가
#     4. LOO 탄소 오차의 시드 간 산포
#
#   선행: [A]  (STAT, shift, carbon_err, ALPHAS, REGIONS 재사용)
#   비용: RF 15회. 40~70분. 시드마다 Drive에 저장하므로 끊겨도 이어짐.
# ============================================================
import os, numpy as np, warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

SEEDS = [16, 17, 18, 19, 20]
CKPT  = f'{OUT}/alpha_seeds'
os.makedirs(CKPT, exist_ok=True)

CROSS = [(a, b) for a in REGIONS for b in REGIONS if a != b]
A_F1  = 0.50                      # F1 개선을 읽을 기준점 ([A]에서 평균 최고)

def run_seed(seed):
    """returns f1[6, nA], carb[6, nA]"""
    f = f'{CKPT}/seed{seed}.npz'
    if os.path.exists(f):
        z = np.load(f); print(f'  seed {seed}: 캐시')
        return z['f1'], z['carb']

    idx = {}
    for rg in REGIONS:
        r = np.random.default_rng(seed)
        tiles = np.unique(S[rg]['tile'])
        te_t  = set(r.choice(tiles, int(len(tiles) * 0.3), replace=False).tolist())
        is_te = np.isin(S[rg]['tile'], list(te_t))
        idx[rg] = (r.choice(np.where(~is_te)[0], min(N_TR, (~is_te).sum()), replace=False),
                   r.choice(len(S[rg]['y']), N_TE, replace=False))

    F = np.zeros((len(CROSS), len(ALPHAS)))
    C_ = np.zeros_like(F)
    for a in REGIONS:
        clf = RandomForestClassifier(n_estimators=80, min_samples_leaf=20,
                                     max_features='sqrt', class_weight='balanced',
                                     n_jobs=-1, random_state=seed)
        clf.fit(S[a]['X'][idx[a][0]], S[a]['y'][idx[a][0]])
        for b in REGIONS:
            if a == b: continue
            i = CROSS.index((a, b)); te = idx[b][1]; y = S[b]['y'][te]
            for j, al in enumerate(ALPHAS):
                p = clf.predict(shift(b, te, a, al))
                F[i, j] = f1_score(y, p, average='macro', labels=[1, 2, 3])
                C_[i, j] = carbon_err(y, p)
        print(f'  seed {seed}: {a[:3]} 완료')
    np.savez(f, f1=F, carb=C_)
    return F, C_

FS, CS = [], []
for s in SEEDS:
    f, c = run_seed(s); FS.append(f); CS.append(c)
FS, CS = np.stack(FS), np.stack(CS)          # (nseed, 6, nA)

def alpha_star(c):
    """탄소 오차 곡선 → 0 통과점"""
    out = np.full(len(CROSS), np.nan)
    for i in range(len(CROSS)):
        y = c[i]
        k = next((t for t in range(len(ALPHAS)-1) if y[t]*y[t+1] < 0), None)
        if k is not None:
            out[i] = ALPHAS[k] + (ALPHAS[k+1]-ALPHAS[k]) * abs(y[k])/(abs(y[k])+abs(y[k+1]))
    return out

AST = np.stack([alpha_star(c) for c in CS])  # (nseed, 6)
nan = int(np.isnan(AST).sum())
if nan:
    print(f'\n★ α* 미검출 {nan}/{AST.size} — 해당 조합은 ALPHAS 구간 안에서 부호가\n  뒤집히지 않았다. ALPHAS 에 1.25, 1.5 를 추가하거나 그 조합을 따로 볼 것')

# ---------- 1. α* 안정성 ----------
print('\n[1] α*  (시드 평균 ± sd)')
print(f'{"조합":12s}' + ''.join(f'{f"s{s}":>8s}' for s in SEEDS) + f'{"평균":>9s}{"sd":>7s}')
for i, (a, b) in enumerate(CROSS):
    v = AST[:, i]
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{x:8.3f}' for x in v)
          + f'{np.nanmean(v):9.3f}{np.nanstd(v):7.3f}')

print('\n[2] 학습 지역별 α*')
print(f'{"학습":10s}{"평균":>9s}{"시드간 sd":>11s}{"조합간 sd":>11s}')
mu_a = {}
for a in REGIONS:
    ii = [i for i, k in enumerate(CROSS) if k[0] == a]
    v = AST[:, ii]                                   # (nseed, 2)
    mu_a[a] = np.nanmean(v)
    print(f'{a:10s}{np.nanmean(v):9.3f}{np.nanstd(np.nanmean(v,1)):11.3f}'
          f'{np.nanmean(np.nanstd(v,1)):11.3f}')

# ---------- 3. 구조가 유지되는가 ----------
print('\n[3] 학습 지역 구조 — 군간/군내 비 (시드별)')
ratios = []
for t, s in enumerate(SEEDS):
    g = [AST[t, [i for i, k in enumerate(CROSS) if k[0] == a]] for a in REGIONS]
    win = np.nanmean([np.nanstd(x) for x in g])
    btw = np.nanstd([np.nanmean(x) for x in g])
    ratios.append(btw / (win + 1e-9))
    print(f'  seed {s}: 군내 {win:.3f} / 군간 {btw:.3f}  → 비 {ratios[-1]:5.1f}')
print(f'  → 모든 시드에서 3 이상이면 구조 확정. 최소 {min(ratios):.1f}')

# ---------- 4. F1 개선 ----------
ja = ALPHAS.index(A_F1); j0 = ALPHAS.index(0.0)
d = FS[:, :, ja] - FS[:, :, j0]
print(f'\n[4] F1 개선 (α={A_F1:g} − α=0)')
print(f'{"조합":12s}' + ''.join(f'{f"s{s}":>8s}' for s in SEEDS) + f'{"평균":>9s}')
for i, (a, b) in enumerate(CROSS):
    print(f'{a[:3]}→{b[:3]:8s}' + ''.join(f'{x:+8.3f}' for x in d[:, i]) + f'{d[:,i].mean():+9.3f}')
print(f'{"전체":12s}' + ' '*(8*len(SEEDS)) + f'{d.mean():+9.3f}')
print(f'  양수 {int((d>0).sum())}/{d.size}   '
      f'시드 내 최소 {d.min():+.3f}   조합별 평균 최소 {d.mean(0).min():+.3f}')
print('  → 30/30 양수면 개선은 확정. 하나라도 음수면 그 조합을 따로 보고할 것')

# ---------- 5. LOO ----------
print('\n[5] LOO 탄소 오차 — b로 α 추정 → c에 적용')
loo = []
for t in range(len(SEEDS)):
    e = []
    for a in REGIONS:
        o = [k[1] for k in CROSS if k[0] == a]
        for b, c in [(o[0], o[1]), (o[1], o[0])]:
            al = AST[t, CROSS.index((a, b))]
            if np.isnan(al): continue
            e.append(abs(np.interp(al, ALPHAS, CS[t, CROSS.index((a, c))])))
    loo.append(np.mean(e))
    print(f'  seed {SEEDS[t]}: {loo[-1]:.3f}%')
lo = np.abs(CS[:, :, j0]).mean(1)
print(f'\n  하한 (보정 없음)  {lo.mean():.3f} ± {lo.std():.3f} %')
print(f'  LOO              {np.mean(loo):.3f} ± {np.std(loo):.3f} %')
print(f'  NOISE (계수 불확실성)  {NOISE:.3f} %p')
print('  → LOO 평균+sd 가 NOISE 수준이면 보정 후 오차가 계수 불확실성에 묻힌다')

  seed 16: dae 완료
  seed 16: hon 완료
  seed 16: sun 완료
  seed 17: dae 완료
  seed 17: hon 완료
  seed 17: sun 완료
  seed 18: dae 완료
  seed 18: hon 완료
  seed 18: sun 완료
  seed 19: dae 완료
  seed 19: hon 완료
  seed 19: sun 완료
  seed 20: dae 완료
  seed 20: hon 완료
  seed 20: sun 완료

[1] α*  (시드 평균 ± sd)
조합               s16     s17     s18     s19     s20       평균     sd
dae→hon        0.576   0.724   0.732   0.710   0.724    0.693  0.059
dae→sun        0.535   0.294   0.423   0.499   0.346    0.419  0.090
hon→dae        0.297   0.284   0.369   0.359   0.343    0.331  0.034
hon→sun        0.335   0.308   0.386   0.369   0.342    0.348  0.027
sun→dae        0.817   0.822   0.818   0.824   0.752    0.806  0.027
sun→hon        0.788   0.805   0.800   0.798   0.779    0.794  0.010

[2] 학습 지역별 α*
학습               평균     시드간 sd     조합간 sd
daejeon       0.556      0.033      0.137
hongcheon     0.339      0.030      0.009
suncheon      0.800      0.018      0.012

[3] 학습 지역 구조 — 군간/군내 비 (시드별)
  seed 16: 군